# Clasificador HAM10000 - Lesiones Cutáneas

Este notebook implementa un sistema de clasificación multiclase de lesiones cutáneas usando el dataset HAM10000.

## Objetivos
- Implementar 4 modelos de deep learning:
  1. Modelo Tabular (datos demográficos)
  2. Modelo CNN (imágenes)
  3. Late Fusion (combinación de predicciones)
  4. Early Fusion (combinación de características)

---

## 1. Setup y Configuración

Importamos las librerías necesarias y configuramos el entorno de Google Colab.

In [ ]:
# Imports necesarios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Concatenate, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
plt.style.use('default')
sns.set_palette('husl')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Verificar disponibilidad de GPU
print("GPU disponible:", tf.config.list_physical_devices('GPU'))
if tf.config.list_physical_devices('GPU'):
    print("✓ Entrenamiento con GPU habilitado")
else:
    print("⚠ Ejecutando en CPU - El entrenamiento será más lento")

### Montar Google Drive

Montamos Google Drive para acceder a los archivos del dataset HAM10000.

In [ ]:
# Montar Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive montado correctamente")
    
    # Definir ruta base del dataset
    # NOTA: Ajustar esta ruta según la ubicación de tus archivos en Google Drive
    BASE_PATH = '/content/drive/MyDrive/HAM10000/'
    
except Exception as e:
    print(f"Error montando Google Drive: {e}")
    print("Asegúrate de estar ejecutando en Google Colab")
    # Para ejecución local, definir ruta alternativa
    BASE_PATH = './practica/data/'

In [ ]:
# Verificar acceso a los archivos del dataset
import os

# Archivos requeridos
IMAGES_FILE = os.path.join(BASE_PATH, 'hmnist_28_28_RGB.csv')
METADATA_FILE = os.path.join(BASE_PATH, 'HAM10000_metadata.csv')

print("Verificando archivos del dataset...")
print(f"Ruta base: {BASE_PATH}")
print(f"\nArchivo de imágenes: {IMAGES_FILE}")
print(f"  Existe: {os.path.exists(IMAGES_FILE)}")
print(f"\nArchivo de metadata: {METADATA_FILE}")
print(f"  Existe: {os.path.exists(METADATA_FILE)}")

if not os.path.exists(IMAGES_FILE) or not os.path.exists(METADATA_FILE):
    print("\n⚠ ERROR: No se encontraron los archivos del dataset")
    print("\nPor favor, asegúrate de:")
    print("1. Haber descargado el dataset HAM10000")
    print("2. Subir los archivos a Google Drive en la carpeta correcta")
    print("3. Ajustar la variable BASE_PATH si es necesario")
else:
    print("\n✓ Todos los archivos encontrados correctamente")

### Funciones de Carga de Datos

Implementamos funciones para cargar las imágenes y los metadatos del dataset.

In [ ]:
def load_images(filepath):
    """
    Carga las imágenes desde el archivo CSV y las convierte a array numpy.
    
    Args:
        filepath (str): Ruta al archivo hmnist_28_28_RGB.csv
    
    Returns:
        tuple: (X_images, y_labels)
            - X_images: array numpy de shape (10015, 28, 28, 3)
            - y_labels: array numpy de shape (10015,) con las etiquetas
    """
    print("Cargando imágenes desde CSV...")
    
    # Leer el archivo CSV
    df = pd.read_csv(filepath)
    
    print(f"  Archivo cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    # La última columna contiene las etiquetas (label)
    y_labels = df['label'].values
    
    # Las demás columnas contienen los píxeles (28*28*3 = 2352 columnas)
    X_flat = df.drop('label', axis=1).values
    
    # Reshape a formato de imagen (N, 28, 28, 3)
    n_samples = X_flat.shape[0]
    X_images = X_flat.reshape(n_samples, 28, 28, 3)
    
    print(f"  Imágenes reshape: {X_images.shape}")
    print(f"  Etiquetas: {y_labels.shape}")
    print(f"  Rango de píxeles: [{X_images.min()}, {X_images.max()}]")
    print(f"  Clases únicas: {np.unique(y_labels)}")
    
    return X_images, y_labels

In [ ]:
def load_metadata(filepath):
    """
    Carga los metadatos del dataset HAM10000.
    
    Args:
        filepath (str): Ruta al archivo HAM10000_metadata.csv
    
    Returns:
        pd.DataFrame: DataFrame con los metadatos
    """
    print("Cargando metadatos desde CSV...")
    
    # Leer el archivo CSV
    df_metadata = pd.read_csv(filepath)
    
    print(f"  Metadatos cargados: {df_metadata.shape[0]} filas, {df_metadata.shape[1]} columnas")
    print(f"  Columnas: {list(df_metadata.columns)}")
    
    return df_metadata

### Cargar Datos

Ejecutamos las funciones de carga y verificamos que los datos se cargaron correctamente.

In [ ]:
# Cargar imágenes
X_images, y_labels = load_images(IMAGES_FILE)

print("\n" + "="*60)
print("RESUMEN DE IMÁGENES CARGADAS")
print("="*60)
print(f"Shape de imágenes: {X_images.shape}")
print(f"Shape de etiquetas: {y_labels.shape}")
print(f"Tipo de datos (imágenes): {X_images.dtype}")
print(f"Tipo de datos (etiquetas): {y_labels.dtype}")

In [ ]:
# Cargar metadatos
df_metadata = load_metadata(METADATA_FILE)

print("\n" + "="*60)
print("RESUMEN DE METADATOS CARGADOS")
print("="*60)
print(f"Shape del DataFrame: {df_metadata.shape}")
print(f"\nPrimeras 5 filas:")
print(df_metadata.head())

In [ ]:
# Información detallada del DataFrame
print("\nInformación del DataFrame:")
print(df_metadata.info())

print("\nEstadísticas descriptivas:")
print(df_metadata.describe())

In [ ]:
# Verificar valores missing
print("\nValores missing por columna:")
missing_values = df_metadata.isnull().sum()
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("✓ No hay valores missing en los metadatos")
else:
    print(f"⚠ Total de valores missing: {missing_values.sum()}")

In [ ]:
# Distribución de clases
print("\nDistribución de clases (dx):")
class_distribution = df_metadata['dx'].value_counts().sort_index()
print(class_distribution)

# Visualizar distribución
plt.figure(figsize=(10, 6))
class_distribution.plot(kind='bar', color='steelblue')
plt.title('Distribución de Clases en el Dataset HAM10000', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de Lesión', fontsize=12)
plt.ylabel('Número de Muestras', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n⚠ Dataset desbalanceado: la clase 'nv' representa {class_distribution['nv']/len(df_metadata)*100:.1f}% del total")

In [ ]:
# Visualizar algunas imágenes de ejemplo
print("\nVisualizando muestras de imágenes...")

# Mapeo de etiquetas numéricas a nombres de clases
label_to_class = {
    0: 'akiec',  # Actinic keratoses
    1: 'bcc',    # Basal cell carcinoma
    2: 'bkl',    # Benign keratosis
    3: 'df',     # Dermatofibroma
    4: 'mel',    # Melanoma
    5: 'nv',     # Melanocytic nevi
    6: 'vasc'    # Vascular lesions
}

# Seleccionar una muestra de cada clase
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()

for i, (label, class_name) in enumerate(label_to_class.items()):
    # Encontrar primera imagen de esta clase
    idx = np.where(y_labels == label)[0][0]
    
    axes[i].imshow(X_images[idx])
    axes[i].set_title(f'Clase {label}: {class_name}', fontsize=10, fontweight='bold')
    axes[i].axis('off')

# Ocultar el último subplot (tenemos 7 clases, no 8)
axes[7].axis('off')

plt.suptitle('Ejemplos de Imágenes por Clase', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Verificación Final

Confirmamos que los datos se cargaron correctamente y están listos para el preprocesamiento.

In [ ]:
# Verificación final de consistencia
print("="*60)
print("VERIFICACIÓN FINAL DE DATOS")
print("="*60)

# Verificar que el número de muestras coincide
assert X_images.shape[0] == len(y_labels), "Inconsistencia entre imágenes y etiquetas"
assert X_images.shape[0] == len(df_metadata), "Inconsistencia entre imágenes y metadatos"

print(f"✓ Número de muestras consistente: {X_images.shape[0]}")

# Verificar shape de imágenes
assert X_images.shape == (10015, 28, 28, 3), f"Shape incorrecto: {X_images.shape}"
print(f"✓ Shape de imágenes correcto: {X_images.shape}")

# Verificar rango de etiquetas
assert y_labels.min() >= 0 and y_labels.max() <= 6, "Etiquetas fuera de rango"
print(f"✓ Etiquetas en rango válido: [0, 6]")

# Verificar que no hay NaN en imágenes
assert not np.isnan(X_images).any(), "NaN encontrado en imágenes"
print(f"✓ No hay valores NaN en imágenes")

print("\n" + "="*60)
print("✓ DATOS CARGADOS CORRECTAMENTE")
print("✓ LISTOS PARA PREPROCESAMIENTO")
print("="*60)